In [ ]:
import time
start_time = time.time()

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Selected device: {device}")


In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()} if isinstance(next(iter(model.config.id2label.keys())), str) else model.config.id2label
label2id = {str(k).upper(): int(v) for k, v in model.config.label2id.items()} if model.config.label2id else {"LABEL_0": 0, "LABEL_1": 1}

print(f"Loaded model: {model_name}")
print(f"Model labels: {model.config.id2label}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Validation examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()
print(preview_df[["sentence1", "sentence2", "label"]])


In [ ]:
batch_size = 64
predictions = []
true_labels = []
all_scores = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]

    encodings = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    encodings = {k: v.to(device) for k, v in encodings.items()}

    with torch.no_grad():
        outputs = model(**encodings)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        batch_preds = torch.argmax(logits, dim=-1)

    predictions.extend(batch_preds.detach().cpu().numpy().tolist())
    true_labels.extend(batch["label"])
    all_scores.extend(probs[:, 1].detach().cpu().numpy().tolist())

print(f"Completed inference for {len(predictions)} examples.")


In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)


In [ ]:
results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "split": "validation",
        "num_examples": len(dataset),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": str(device),
        "batch_size": batch_size
    }
])

print(results_df.to_string(index=False))


In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["predicted_label"] = predictions
examples_df["paraphrase_score"] = all_scores

print(examples_df.head(10).to_string(index=False))

mismatches_df = examples_df[examples_df["true_label"] != examples_df["predicted_label"]].copy()
print(f"\nMismatches: {len(mismatches_df)}")
if len(mismatches_df) > 0:
    print(mismatches_df.head(10).to_string(index=False))


In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")
